<a href="https://colab.research.google.com/github/BWMIN-Hub/SR_practice/blob/main/notebooks/07_hat_x3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HAT ×3 — Transformer 를 더 키우면

흐린 위성사진(10 m) → 3배 선명하게(3.33 m).

## 1. 데이터

In [ ]:
import sys, urllib.request
!pip install -q timm einops

LIB = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main/lib'
for m in ['sr_utils.py', 'hat_arch.py', 'hat_models.py']:
    urllib.request.urlretrieve(f'{LIB}/{m}', m)
    sys.modules.pop(m[:-3], None)     # 이미 불러온 옛 모듈이 남아 있으면 비운다

from sr_utils import *

show_data()        # validation 2패치 + test 2구역

## 2. 훈련

**코드가 도는지 확인하는 용도다.** 16장으로 1 epoch 만 돌린다.
아래 결과는 전체 데이터로 학습해둔 가중치를 쓴다.

SwinIR 보다 무거워서 128px 을 그대로 넣으면 12 GB 를 쓴다. Colab T4 로는 빠듯하다.
64px 씩 잘라 batch 1 로 돌린다 (약 3 GB). 실제 학습도 48px 씩 잘라서 했다.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from hat_models import build_hat

# 자를 크기는 창 크기 16 의 배수여야 한다. HAT 이 입력을 16×16 창으로 쪼개 보기 때문이다.
N_TRAIN, EPOCHS, BATCH, CROP = 16, 1, 1, 64
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

lo, hi = zip(*[pair('training', s) for s in list_split('training')[:N_TRAIN]])
to_t = lambda a: torch.from_numpy(np.stack(a).transpose(0, 3, 1, 2)).float() / 255
x, y = to_t(lo)[:, :, :CROP, :CROP], to_t(hi)[:, :, :CROP * 3, :CROP * 3]
loader = DataLoader(TensorDataset(x, y), batch_size=BATCH, shuffle=True)

net = build_hat(3).to(dev).train()
opt = torch.optim.Adam(net.parameters(), 2e-4, betas=(0.9, 0.99))
crit = nn.L1Loss()

for ep in range(1, EPOCHS + 1):
    tot = 0.0
    for xb, yb in loader:
        loss = crit(net(xb.to(dev)), yb.to(dev))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
        tot += loss.item()
    print(f'epoch {ep}/{EPOCHS}   L1 {tot/len(loader):.5f}')

del net, opt, loader, x, y
torch.cuda.empty_cache()      # 다음 셀에서 학습된 가중치를 올릴 자리를 비운다

## 3. 결과

In [ ]:
MODEL = f'{BASE}/models/07_hat_x3'

from hat_models import load_hat, hat_upscale

net = load_hat(fetch(f'{MODEL}/checkpoints/hat_x3.pth', 'hat_x3.pth'))
print(f'HAT  {sum(p.numel() for p in net.parameters())/1e6:.2f}M')

upscale = lambda lr: hat_upscale(net, lr)

show_results(upscale, 'HAT', center=[(67, 370), (313, 36)])

## 4. 최종 테스트

정답이 없는 실제 Sentinel-2 촬영본이다. 눈으로 확인한다.


In [ ]:
show_test(upscale, 'HAT', center=(800, 800), size=70)